In [13]:
#from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [14]:
PROJECT_DIR = Path.cwd().parent
output_file = PROJECT_DIR / "data_processed" / "flatfox_rentals_zentralschweiz.csv"
gpkg_path = PROJECT_DIR / "geodata" / "swissBOUNDARIES3D_1_5_LV95_LN02.gpkg"

In [15]:
def merge_flatfox_csvs(project_dir):
    """Liest alle Erhebungs-CSVs ein, entfernt identische Duplikate und behält die neueste Zeile je Inserat-ID."""
    
    project_dir = Path(project_dir)
    source_files = sorted((project_dir / "flatfox_data").glob("*/listings_all_fields.csv"))
    output_file = (project_dir / "data_processed" / "flatfox_listings_merged.csv")

    if not source_files:
        raise FileNotFoundError("Keine Erhebungsdateien gefunden.")

    frames = []
    report = []

    for source in source_files:
        frame = pd.read_csv(
            source,
            encoding="utf-8-sig",
            dtype="string",
            keep_default_na=False,
            na_values=[""],
        )

        frame["pk"] = frame["pk"].str.strip()

        if (frame["pk"].isna() | frame["pk"].eq("")).any():
            raise ValueError(f"Leere Inserat-ID in {source}")

        rows_read = len(frame)
        frame = frame.drop_duplicates()

        if frame["pk"].duplicated().any():
            raise ValueError(
                f"Widersprüchliche Zeilen mit gleicher pk in {source}"
            )

        report.append({
            "source_run": source.parent.name,
            "rows_read": rows_read,
            "unique_ids": len(frame),
            "identical_duplicates": rows_read - len(frame),
        })

        frame["source_run"] = source.parent.name
        frame["source_file"] = source.relative_to(project_dir).as_posix()
        frame["__run_at"] = pd.to_datetime(
            source.parent.name,
            format="%Y%m%dT%H%M%S_%fZ",
            utc=True,
        )
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined = combined.sort_values("__run_at", kind="stable")

    if combined.empty:
        raise ValueError("Die Erhebungsdateien enthalten keine Inserate.")

    # Sichtungen über alle vorhandenen Erhebungen bestimmen.
    grouped = combined.groupby("pk")

    combined["first_seen_at"] = grouped["__run_at"].transform("min")
    combined["last_seen_at"] = grouped["__run_at"].transform("max")
    combined["scrape_count"] = grouped["source_run"].transform("nunique")

    # Je ID die gesamte neueste Zeile übernehmen.
    merged = (
        combined.drop_duplicates("pk", keep="last")
        .drop(columns="__run_at")
        .sort_values("pk")
        .reset_index(drop=True)
    )

    for column in ["first_seen_at", "last_seen_at"]:
        merged[column] = merged[column].dt.strftime(
            "%Y-%m-%dT%H:%M:%S.%fZ"
        )

    # Fertige Datei schreiben und danach das bisherige Ergebnis ersetzen.
    output_file.parent.mkdir(parents=True, exist_ok=True)
    temporary_file = output_file.with_suffix(".csv.tmp")
    merged.to_csv(temporary_file, index=False, encoding="utf-8-sig")
    temporary_file.replace(output_file)

    report_df = pd.DataFrame(report)

    print(f"Erhebungsdateien: {len(source_files)}")
    print(f"Eingelesene Zeilen: {report_df['rows_read'].sum()}")
    print(f"Eindeutige Inserate: {len(merged)}")
    print(f"Gespeichert: {output_file}")

    return merged, report_df

In [16]:
def filter_flatfox_rentals(df, gpkg_path, output_file):
    """Ordnet den übergebenen Inseraten anhand ihrer Koordinaten einen Kanton zu und dokumentiert den Zuordnungsstatus."""
    
    gpkg_path = Path(gpkg_path)
    output_file = Path(output_file)

    if not gpkg_path.is_file():
        raise FileNotFoundError(f"GeoPackage-Datei nicht gefunden: {gpkg_path}")

    df_rentals_geo = df.copy().reset_index(drop=True)

    # Kantonsflächen laden
    layers = gpd.list_layers(gpkg_path)

    canton_layers = layers.loc[layers["name"].str.contains("kantonsgebiet", case=False, na=False),"name",].tolist()

    if len(canton_layers) != 1:
        raise ValueError(
            f"Kantons-Layer nicht eindeutig gefunden: "
            f"{layers['name'].tolist()}"
        )

    cantons = gpd.read_file(gpkg_path, layer=canton_layers[0])

    cantons = cantons.rename(columns={column: column.upper() for column in cantons.columns if column != cantons.geometry.name})

    if cantons.geometry.name != "geometry":
        cantons = cantons.rename_geometry("geometry")

    cantons = cantons.loc[cantons["ICC"].eq("CH")].copy()

    canton_codes = dict(enumerate(
        [
            "ZH", "BE", "LU", "UR", "SZ", "OW", "NW", "GL", "ZG",
            "FR", "SO", "BS", "BL", "SH", "AR", "AI", "SG", "GR",
            "AG", "TG", "TI", "VD", "VS", "NE", "GE", "JU",
        ],
        start=1,
    ))

    cantons["kanton_code"] = pd.to_numeric(cantons["KANTONSNUMMER"], errors="coerce").map(canton_codes)

    if cantons["kanton_code"].isna().any():
        raise ValueError("Unbekannte Kantonsnummer in den Geodaten.")


    # Inserat-Koordinaten vorbereiten
    longitude = pd.to_numeric(df_rentals_geo["longitude"], errors="coerce")
    latitude = pd.to_numeric(df_rentals_geo["latitude"], errors="coerce")

    valid_coordinates = (longitude.between(-180, 180) & latitude.between(-90, 90)).fillna(False)

    df_rentals_geo["kanton_geo"] = pd.Series(pd.NA,index=df_rentals_geo.index,dtype="string",)
    df_rentals_geo["kanton_geo_status"] = "koordinaten_ungueltig"


    # Kantone anhand der Koordinaten bestimmen
    if valid_coordinates.any():
        points = gpd.GeoDataFrame(
            index=df_rentals_geo.index[valid_coordinates],
            geometry=gpd.points_from_xy(
                longitude.loc[valid_coordinates],
                latitude.loc[valid_coordinates],
            ),
            crs="EPSG:4326",
        ).to_crs(cantons.crs)

        matches = gpd.sjoin(
            points,
            cantons[["kanton_code", "geometry"]],
            how="left",
            predicate="intersects",
        )

        candidates = matches.groupby(level=0)["kanton_code"].agg(
            lambda values: tuple(sorted(set(values.dropna())))
        )

        assigned = candidates.map(
            lambda values: values[0] if len(values) == 1 else pd.NA
        )

        df_rentals_geo["kanton_geo"] = (assigned.reindex(df_rentals_geo.index).astype("string"))

        df_rentals_geo.loc[valid_coordinates, "kanton_geo_status"] = "kein_treffer"

        ambiguous_indices = candidates[candidates.map(len).gt(1)].index

        df_rentals_geo.loc[ambiguous_indices, "kanton_geo_status"] = "mehrdeutiger_treffer"
        df_rentals_geo.loc[df_rentals_geo["kanton_geo"].notna(),"kanton_geo_status",] = "zugeordnet"


    # Auf die Zentralschweiz eingrenzen
    central_cantons = ["LU", "UR", "SZ", "OW", "NW", "ZG"]

    df_central = df_rentals_geo.loc[df_rentals_geo["kanton_geo"].isin(central_cantons)].copy()

    # Ergebnis als CSV speichern
    output_file.parent.mkdir(parents=True, exist_ok=True)

    df_central.to_csv(output_file, index=False, encoding="utf-8-sig",)

    return df_rentals_geo, df_central

In [17]:
df_merged, merge_report = merge_flatfox_csvs(PROJECT_DIR)

# Filter der Rohdaten
mask = (df_merged["offer_type"].eq("RENT") & df_merged["object_category"].isin(["APARTMENT", "HOUSE"])).fillna(False)

df_filtered = df_merged.loc[mask]

# Kantone zuordnen und Zentralschweizer Inserate speichern
df_rentals_geo, df_central = filter_flatfox_rentals(df_filtered, gpkg_path, output_file,)

display(merge_report)

print(f"\nZusammengeführte Inserate: {len(df_merged)}")
print(f"Mietwohnungen und Häuser: {len(df_rentals_geo)}")
print(f"Davon Zentralschweiz: {len(df_central)}")
print(f"CSV gespeichert: {output_file}")

print("\nStatus der Kantonszuordnung:")
display(df_rentals_geo["kanton_geo_status"].value_counts())

print("\nInserate pro Zentralschweizer Kanton:")

central_cantons = ["LU", "UR", "SZ", "OW", "NW", "ZG"]

display(df_central["kanton_geo"].value_counts().reindex(central_cantons, fill_value=0))

Erhebungsdateien: 9
Eingelesene Zeilen: 22969
Eindeutige Inserate: 3351
Gespeichert: /Users/enrue/Developer/jupyter-setup/cip_projekt/data_processed/flatfox_listings_merged.csv


,source_run,rows_read,unique_ids,identical_duplicates
0,20261018T100611_875376Z,284,284,0
1,20261018T103104_510890Z,1509,1509,0
2,20261018T121452_635701Z,3008,3008,0
3,20261018T151426_580437Z,3008,3008,0
4,20261018T191116_940418Z,3017,3017,0
5,20261019T151108_866230Z,3022,3022,0
6,20261020T191239_840560Z,3049,3049,0
7,20261021T192550_227446Z,3035,3035,0
8,20261022T153515_050865Z,3037,3037,0



Zusammengeführte Inserate: 3351
Mietwohnungen und Häuser: 1554
Davon Zentralschweiz: 868
CSV gespeichert: /Users/enrue/Developer/jupyter-setup/cip_projekt/data_processed/flatfox_rentals_zentralschweiz.csv

Status der Kantonszuordnung:


kanton_geo_status
zugeordnet    1554
Name: count, dtype: int64


Inserate pro Zentralschweizer Kanton:


kanton_geo
LU    587
UR     19
SZ    110
OW     22
NW     12
ZG    118
Name: count, dtype: Int64